# WAN multiview sweep analysis

This notebook:
- scans run folders under `Open-Sora/outputs`
- reads each `eval_metrics.jsonl`
- keeps **only** runs that contain at least one line with `actual_update_step == 11750`
- aggregates metrics/configs
- reports best settings and ablations for:
  - `discriminator_choice` (`none`, `Train`, `TrainMultiview4D`)
  - `gen_disc_weight` (for GAN runs)
  - `perceptual_loss_weight`
  - `kl_loss_weight`

Primary ranking metric is `psnr` (higher better), with `ssim` and `mse` as secondary checks.

In [2]:
!pip install pandas -q

In [13]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Any

import pandas as pd

# -------- Config --------
OUTPUTS_ROOT = Path('/home/piado/projects/aip-lindell/piado/vae/Open-Sora/outputs')
TARGET_STEP = 11750

print(f'Outputs root: {OUTPUTS_ROOT}')
print(f'Target step: {TARGET_STEP}')
print(f'Exists: {OUTPUTS_ROOT.exists()}')

Outputs root: /home/piado/projects/aip-lindell/piado/vae/Open-Sora/outputs
Target step: 11750
Exists: True


In [14]:
def _to_float(x: Any):
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return None


def _safe_json_lines(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows


def _extract_metrics(rec: dict[str, Any]) -> dict[str, Any]:
    """Normalize available metrics to psnr/ssim/mse fields."""
    m = rec.get('metrics', {}) or {}
    return {
        'psnr': m.get('psnr_mean', m.get('psnr')),
        'ssim': m.get('ssim_mean', m.get('ssim')),
        'mse': m.get('mse_mean', m.get('mse')),
        'psnr_std': m.get('psnr_std'),
        'ssim_std': m.get('ssim_std'),
        'mse_std': m.get('mse_std'),
    }


def strip_job_suffix(folder_name: str) -> str:
    """Strip trailing ``__job<digits>_t<digits>`` from sweep folder names for display."""
    return re.sub(r'__job\d+_t\d+$', '', folder_name)


def _extract_config(rec: dict[str, Any], folder_name: str) -> dict[str, Any]:
    lc = rec.get('loss_config', {}) or {}

    # Fallback parsing from folder name if missing in loss_config.
    # Example: sweep_train__perc3p0__d005__k1em6__job3154659_t34
    disc = lc.get('discriminator_choice')
    if disc is None:
        if 'sweep_none__' in folder_name:
            disc = 'none'
        elif 'sweep_train__' in folder_name:
            disc = 'Train'
        elif 'sweep_mv4d__' in folder_name:
            disc = 'TrainMultiview4D'

    return {
        'discriminator_choice': disc,
        'gen_disc_weight': _to_float(lc.get('gen_disc_weight')),
        'perceptual_loss_weight': _to_float(lc.get('perceptual_loss_weight')),
        'kl_loss_weight': _to_float(lc.get('kl_loss_weight')),
        'vae_loss_preset': lc.get('vae_loss_preset'),
    }


def choose_record_at_target_step(records: list[dict[str, Any]], target_step: int):
    """
    Pick one record at target step with preference:
    1) kind=full_eval and split=val
    2) any full_eval
    3) train_batch
    4) first available at target step
    """
    at_step = [r for r in records if r.get('actual_update_step') == target_step]
    if not at_step:
        return None

    for r in at_step:
        if r.get('kind') == 'full_eval' and r.get('split') == 'val':
            return r
    for r in at_step:
        if r.get('kind') == 'full_eval':
            return r
    for r in at_step:
        if r.get('kind') == 'train_batch':
            return r
    return at_step[0]

In [15]:
rows = []
missing_eval = []
missing_target_step = []

for run_dir in sorted(p for p in OUTPUTS_ROOT.iterdir() if p.is_dir()):
    eval_path = run_dir / 'eval_metrics.jsonl'
    if not eval_path.exists():
        missing_eval.append(run_dir.name)
        continue

    records = _safe_json_lines(eval_path)
    chosen = choose_record_at_target_step(records, TARGET_STEP)
    if chosen is None:
        missing_target_step.append(run_dir.name)
        continue

    metrics = _extract_metrics(chosen)
    cfg = _extract_config(chosen, run_dir.name)

    row = {
        'run_dir': run_dir.name,
        'run_label': strip_job_suffix(run_dir.name),
        'eval_path': str(eval_path),
        'kind': chosen.get('kind'),
        'split': chosen.get('split'),
        'actual_update_step': chosen.get('actual_update_step'),
        'global_step': chosen.get('global_step'),
        'epoch': chosen.get('epoch'),
        **metrics,
        **cfg,
    }
    rows.append(row)

df = pd.DataFrame(rows)

if df.empty:
    print('No runs matched the target step filter.')
else:
    for c in ['psnr', 'ssim', 'mse', 'gen_disc_weight', 'perceptual_loss_weight', 'kl_loss_weight']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    # Normalize discriminator spelling
    df['discriminator_choice'] = df['discriminator_choice'].replace({
        'None': 'none',
        'train': 'Train',
        'mv4d': 'TrainMultiview4D',
    })

print(f'Total output dirs: {sum(1 for p in OUTPUTS_ROOT.iterdir() if p.is_dir())}')
print(f'Included runs (step={TARGET_STEP} present): {len(df)}')
print(f'Missing eval_metrics.jsonl: {len(missing_eval)}')
print(f'Missing target step ({TARGET_STEP}): {len(missing_target_step)}')

_preview = df
if not df.empty:
    _preview = df[
        ['run_label'] + [c for c in df.columns if c not in ('run_label', 'run_dir')]
    ]
_preview.head(10)

Total output dirs: 195
Included runs (step=11750 present): 92
Missing eval_metrics.jsonl: 14
Missing target step (11750): 89


,run_label,eval_path,kind,split,actual_update_step,global_step,epoch,psnr,ssim,mse,psnr_std,ssim_std,mse_std,discriminator_choice,gen_disc_weight,perceptual_loss_weight,kl_loss_weight,vae_loss_preset
0,sweep_mv4d__perc1p5__d005__k1em6,/home/piado/projects/aip-lindell/piado/vae/Ope...,full_eval,val,11750,11749,29,28.70000,0.996094,0.001408,1.233896,0.001235,0.000347,TrainMultiview4D,0.1,1.5,1.000000e-06,default
1,sweep_mv4d__perc1p5__d005__k1em7,/home/piado/projects/aip-lindell/piado/vae/Ope...,full_eval,val,11750,11749,29,28.73750,0.995898,0.001388,1.205392,0.000851,0.000333,TrainMultiview4D,0.1,1.5,1.000000e-07,default
2,sweep_mv4d__perc1p5__d005__k1em8,/home/piado/projects/aip-lindell/piado/vae/Ope...,full_eval,val,11750,11749,29,28.63750,0.995898,0.001429,1.346233,0.000851,0.000396,TrainMultiview4D,0.1,1.5,1.000000e-08,default
3,sweep_mv4d__perc1p5__d01__k1em6,/home/piado/projects/aip-lindell/piado/vae/Ope...,full_eval,val,11750,11749,29,28.67500,0.995898,0.001420,1.265775,0.000851,0.000371,TrainMultiview4D,0.1,1.5,1.000000e-06,default
4,sweep_mv4d__perc1p5__d01__k1em7,/home/piado/projects/aip-lindell/piado/vae/Ope...,full_eval,val,11750,11749,29,28.23750,0.995313,0.001548,1.144894,0.001563,0.000348,TrainMultiview4D,0.1,1.5,1.000000e-07,default
5,sweep_mv4d__perc1p5__d01__k1em8,/home/piado/projects/aip-lindell/piado/vae/Ope...,full_eval,val,11750,11749,29,26.41875,0.993555,0.002341,0.921340,0.002554,0.000364,TrainMultiview4D,0.1,1.5,1.000000e-08,default
6,sweep_mv4d__perc1p5__d02__k1em6,/home/piado/projects/aip-lindell/piado/vae/Ope...,full_eval,val,11750,11749,29,28.47500,0.995703,0.001504,1.437011,0.002439,0.000413,TrainMultiview4D,0.1,1.5,1.000000e-06,default
7,sweep_mv4d__perc1p5__d02__k1em7,/home/piado/projects/aip-lindell/piado/vae/Ope...,full_eval,val,11750,11749,29,27.75625,0.995313,0.001746,1.223772,0.001563,0.000435,TrainMultiview4D,0.1,1.5,1.000000e-07,default
8,sweep_mv4d__perc1p5__d02__k1em8,/home/piado/projects/aip-lindell/piado/vae/Ope...,full_eval,val,11750,11749,29,28.71875,0.995703,0.001400,1.286271,0.002104,0.000372,TrainMultiview4D,0.1,1.5,1.000000e-08,default
9,sweep_mv4d__perc1p5__d03__k1em6,/home/piado/projects/aip-lindell/piado/vae/Ope...,full_eval,val,11750,11749,29,27.30625,0.994922,0.001902,0.913847,0.001790,0.000328,TrainMultiview4D,0.1,1.5,1.000000e-06,default


In [16]:
if not df.empty:
    sort_cols = ['psnr', 'ssim', 'mse']
    available = [c for c in sort_cols if c in df.columns]

    ranked = df.sort_values(
        by=['psnr', 'ssim', 'mse'],
        ascending=[False, False, True],
        na_position='last'
    )

    print('Top 15 runs overall (ranked by psnr desc, ssim desc, mse asc):')
    display_cols = [
        'run_label', 'discriminator_choice', 'gen_disc_weight',
        'perceptual_loss_weight', 'kl_loss_weight',
        'psnr', 'ssim', 'mse', 'kind', 'split'
    ]
    display(ranked[display_cols].head(15))

    best = ranked.iloc[0]
    print('\nBest config overall:')
    print(best[display_cols].to_string())

Top 15 runs overall (ranked by psnr desc, ssim desc, mse asc):


,run_label,discriminator_choice,gen_disc_weight,perceptual_loss_weight,kl_loss_weight,psnr,ssim,mse,kind,split
47,sweep_none__perc1p5__k1em6,none,NaN,1.5,1.000000e-06,28.96250,0.996289,0.001330,full_eval,val
50,sweep_none__perc2p0__k1em6,none,NaN,2.0,1.000000e-06,28.85000,0.996289,0.001369,full_eval,val
10,sweep_mv4d__perc1p5__d03__k1em7,TrainMultiview4D,0.1,1.5,1.000000e-07,28.75625,0.995898,0.001386,full_eval,val
1,sweep_mv4d__perc1p5__d005__k1em7,TrainMultiview4D,0.1,1.5,1.000000e-07,28.73750,0.995898,0.001388,full_eval,val
8,sweep_mv4d__perc1p5__d02__k1em8,TrainMultiview4D,0.1,1.5,1.000000e-08,28.71875,0.995703,0.001400,full_eval,val
0,sweep_mv4d__perc1p5__d005__k1em6,TrainMultiview4D,0.1,1.5,1.000000e-06,28.70000,0.996094,0.001408,full_eval,val
3,sweep_mv4d__perc1p5__d01__k1em6,TrainMultiview4D,0.1,1.5,1.000000e-06,28.67500,0.995898,0.001420,full_eval,val
48,sweep_none__perc1p5__k1em7,none,NaN,1.5,1.000000e-07,28.66875,0.995508,0.001411,full_eval,val
20,sweep_mv4d__perc2p0__d02__k1em8,TrainMultiview4D,0.1,2.0,1.000000e-08,28.66250,0.996289,0.001411,full_eval,val
49,sweep_none__perc1p5__k1em8,none,NaN,1.5,1.000000e-08,28.65000,0.996094,0.001421,full_eval,val



Best config overall:
run_label                 sweep_none__perc1p5__k1em6
discriminator_choice                            none
gen_disc_weight                                  NaN
perceptual_loss_weight                           1.5
kl_loss_weight                              0.000001
psnr                                         28.9625
ssim                                        0.996289
mse                                          0.00133
kind                                       full_eval
split                                            val


In [17]:
def summarize_factor(frame: pd.DataFrame, factor: str) -> pd.DataFrame:
    agg = (
        frame.groupby(factor, dropna=False)
        .agg(
            runs=('run_dir', 'count'),
            psnr_mean=('psnr', 'mean'),
            psnr_max=('psnr', 'max'),
            ssim_mean=('ssim', 'mean'),
            ssim_max=('ssim', 'max'),
            mse_mean=('mse', 'mean'),
            mse_min=('mse', 'min'),
        )
        .reset_index()
        .sort_values(['psnr_mean', 'ssim_mean', 'mse_mean'], ascending=[False, False, True])
    )
    return agg

if not df.empty:
    print('Ablation: discriminator_choice')
    display(summarize_factor(df, 'discriminator_choice'))

    print('Ablation: perceptual_loss_weight')
    display(summarize_factor(df, 'perceptual_loss_weight'))

    print('Ablation: kl_loss_weight')
    display(summarize_factor(df, 'kl_loss_weight'))

    gan_df = df[df['discriminator_choice'].isin(['Train', 'TrainMultiview4D'])].copy()
    if not gan_df.empty:
        print('Ablation: gen_disc_weight (GAN runs only)')
        display(summarize_factor(gan_df, 'gen_disc_weight'))

Ablation: discriminator_choice


,discriminator_choice,runs,psnr_mean,psnr_max,ssim_mean,ssim_max,mse_mean,mse_min
2,none,9,28.261111,28.96250,0.995464,0.996289,0.001573,0.001330
1,TrainMultiview4D,47,27.611968,28.75625,0.994943,0.996289,0.001832,0.001386
0,Train,36,26.819444,28.62500,0.993972,0.995508,0.002232,0.001430


Ablation: perceptual_loss_weight


,perceptual_loss_weight,runs,psnr_mean,psnr_max,ssim_mean,ssim_max,mse_mean,mse_min
0,1.5,27,27.804630,28.96250,0.995059,0.996289,0.001777,0.001330
1,2.0,27,27.315278,28.85000,0.994596,0.996289,0.001978,0.001369
2,3.0,38,27.088816,28.45625,0.994310,0.996094,0.002085,0.001494


Ablation: kl_loss_weight


,kl_loss_weight,runs,psnr_mean,psnr_max,ssim_mean,ssim_max,mse_mean,mse_min
0,1.000000e-08,31,27.432661,28.71875,0.994758,0.996289,0.001911,0.001400
1,1.000000e-07,31,27.376411,28.75625,0.994468,0.996094,0.001991,0.001386
2,1.000000e-06,30,27.284375,28.96250,0.994616,0.996289,0.001988,0.001330


Ablation: gen_disc_weight (GAN runs only)


,gen_disc_weight,runs,psnr_mean,psnr_max,ssim_mean,ssim_max,mse_mean,mse_min
0,0.1,83,27.268223,28.75625,0.994522,0.996289,0.002006,0.001386


In [21]:
if not df.empty:
    print('Best run per discriminator_choice')
    idx = df.groupby('discriminator_choice')['psnr'].idxmax()
    best_per_disc = df.loc[idx].sort_values('psnr', ascending=False)
    display(best_per_disc[[
        'discriminator_choice', 'run_label', 'gen_disc_weight',
        'perceptual_loss_weight', 'kl_loss_weight',
        'psnr', 'ssim', 'mse', 'kind', 'split'
    ]])

    print('Best run per KL within each discriminator')
    best_per_disc_kl = (
        df.sort_values(['discriminator_choice', 'kl_loss_weight', 'psnr', 'ssim', 'mse'], ascending=[True, True, False, False, True])
          .groupby(['discriminator_choice', 'kl_loss_weight'], as_index=False)
          .first()
          .sort_values(['discriminator_choice', 'psnr'], ascending=[True, False])
    )
    display(best_per_disc_kl[[
        'discriminator_choice', 'kl_loss_weight', 'run_label',
        'gen_disc_weight', 'perceptual_loss_weight',
        'psnr', 'ssim', 'mse'
    ]])

Best run per discriminator_choice


,discriminator_choice,run_label,gen_disc_weight,perceptual_loss_weight,kl_loss_weight,psnr,ssim,mse,kind,split
47,none,sweep_none__perc1p5__k1em6,NaN,1.5,1.000000e-06,28.96250,0.996289,0.001330,full_eval,val
10,TrainMultiview4D,sweep_mv4d__perc1p5__d03__k1em7,0.1,1.5,1.000000e-07,28.75625,0.995898,0.001386,full_eval,val
63,Train,sweep_train__perc1p5__d02__k1em7,0.1,1.5,1.000000e-07,28.62500,0.995508,0.001430,full_eval,val


Best run per KL within each discriminator


,discriminator_choice,kl_loss_weight,run_label,gen_disc_weight,perceptual_loss_weight,psnr,ssim,mse
1,Train,1.000000e-07,sweep_train__perc1p5__d02__k1em7,0.1,1.5,28.62500,0.995508,0.001430
0,Train,1.000000e-08,sweep_train__perc1p5__d03__k1em8,0.1,1.5,28.38750,0.995117,0.001511
2,Train,1.000000e-06,sweep_train__perc2p0__d005__k1em6,0.1,2.0,27.88125,0.994922,0.001697
4,TrainMultiview4D,1.000000e-07,sweep_mv4d__perc1p5__d03__k1em7,0.1,1.5,28.75625,0.995898,0.001386
3,TrainMultiview4D,1.000000e-08,sweep_mv4d__perc1p5__d02__k1em8,0.1,1.5,28.71875,0.995703,0.001400
5,TrainMultiview4D,1.000000e-06,sweep_mv4d__perc1p5__d005__k1em6,0.1,1.5,28.70000,0.996094,0.001408
8,none,1.000000e-06,sweep_none__perc1p5__k1em6,NaN,1.5,28.96250,0.996289,0.001330
7,none,1.000000e-07,sweep_none__perc1p5__k1em7,NaN,1.5,28.66875,0.995508,0.001411
6,none,1.000000e-08,sweep_none__perc1p5__k1em8,NaN,1.5,28.65000,0.996094,0.001421


In [19]:
if not df.empty:
    # Interaction table: mean PSNR by discriminator x KL
    pivot_disc_kl = pd.pivot_table(
        df,
        index='discriminator_choice',
        columns='kl_loss_weight',
        values='psnr',
        aggfunc='mean'
    )
    print('Mean PSNR: discriminator_choice x kl_loss_weight')
    display(pivot_disc_kl)

    # Interaction table: mean PSNR by discriminator x perceptual
    pivot_disc_perc = pd.pivot_table(
        df,
        index='discriminator_choice',
        columns='perceptual_loss_weight',
        values='psnr',
        aggfunc='mean'
    )
    print('Mean PSNR: discriminator_choice x perceptual_loss_weight')
    display(pivot_disc_perc)

    # Interaction table for GAN only: mean PSNR by discriminator x gen_disc_weight
    gan_df = df[df['discriminator_choice'].isin(['Train', 'TrainMultiview4D'])]
    if not gan_df.empty:
        pivot_disc_gdw = pd.pivot_table(
            gan_df,
            index='discriminator_choice',
            columns='gen_disc_weight',
            values='psnr',
            aggfunc='mean'
        )
        print('Mean PSNR: GAN discriminator_choice x gen_disc_weight')
        display(pivot_disc_gdw)

Mean PSNR: discriminator_choice x kl_loss_weight


kl_loss_weight,1.000000e-08,1.000000e-07,1.000000e-06
discriminator_choice,,,
Train,27.082812,26.590625,26.784896
TrainMultiview4D,27.550391,27.778516,27.500000
none,28.204167,28.375000,28.204167


Mean PSNR: discriminator_choice x perceptual_loss_weight


perceptual_loss_weight,1.5,2.0,3.0
discriminator_choice,,,
Train,27.158333,26.748437,26.551562
TrainMultiview4D,28.211979,27.556771,27.327717
none,28.760417,28.616667,27.406250


Mean PSNR: GAN discriminator_choice x gen_disc_weight


gen_disc_weight,0.1
discriminator_choice,
Train,26.819444
TrainMultiview4D,27.611968


In [20]:
if not df.empty:
    # Compact textual conclusions from the aggregated tables.
    disc_summary = summarize_factor(df, 'discriminator_choice')
    perc_summary = summarize_factor(df, 'perceptual_loss_weight')
    kl_summary = summarize_factor(df, 'kl_loss_weight')

    gan_df = df[df['discriminator_choice'].isin(['Train', 'TrainMultiview4D'])].copy()
    gdw_summary = summarize_factor(gan_df, 'gen_disc_weight') if not gan_df.empty else None

    print('=== Auto-summary (based on mean PSNR, with SSIM/MSE tie-breaks) ===')
    print(f"Best discriminator_choice: {disc_summary.iloc[0]['discriminator_choice']}")
    print(f"Best perceptual_loss_weight: {perc_summary.iloc[0]['perceptual_loss_weight']}")
    print(f"Best kl_loss_weight: {kl_summary.iloc[0]['kl_loss_weight']}")
    if gdw_summary is not None and not gdw_summary.empty:
        print(f"Best gen_disc_weight (GAN-only): {gdw_summary.iloc[0]['gen_disc_weight']}")

    ranked = df.sort_values(by=['psnr', 'ssim', 'mse'], ascending=[False, False, True])
    print('\nTop 5 runs:')
    display(ranked[[
        'run_label', 'discriminator_choice', 'gen_disc_weight',
        'perceptual_loss_weight', 'kl_loss_weight',
        'psnr', 'ssim', 'mse'
    ]].head(5))

=== Auto-summary (based on mean PSNR, with SSIM/MSE tie-breaks) ===
Best discriminator_choice: none
Best perceptual_loss_weight: 1.5
Best kl_loss_weight: 1e-08
Best gen_disc_weight (GAN-only): 0.1

Top 5 runs:


,run_label,discriminator_choice,gen_disc_weight,perceptual_loss_weight,kl_loss_weight,psnr,ssim,mse
47,sweep_none__perc1p5__k1em6,none,NaN,1.5,1.000000e-06,28.96250,0.996289,0.001330
50,sweep_none__perc2p0__k1em6,none,NaN,2.0,1.000000e-06,28.85000,0.996289,0.001369
10,sweep_mv4d__perc1p5__d03__k1em7,TrainMultiview4D,0.1,1.5,1.000000e-07,28.75625,0.995898,0.001386
1,sweep_mv4d__perc1p5__d005__k1em7,TrainMultiview4D,0.1,1.5,1.000000e-07,28.73750,0.995898,0.001388
8,sweep_mv4d__perc1p5__d02__k1em8,TrainMultiview4D,0.1,1.5,1.000000e-08,28.71875,0.995703,0.001400


## Visual comparisons at target step

This section augments the tabular sweep summary with actual reconstruction images from each run's `wandb/latest-run/files/media/images` folder.

- Uses `TARGET_STEP` images when available (prefers `val_reconstructions`, then `train_reconstructions`).
- Falls back to nearest available step if exact step image is missing.
- Shows **best vs worst** overall and **best run per ablation value**.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg


def _extract_step_from_image_name(path: Path):
    m = re.search(r"_(\d+)_", path.name)
    return int(m.group(1)) if m else None


def _find_media_image_for_row(row, target_step: int):
    eval_path = Path(row["eval_path"])
    run_dir = eval_path.parent
    images_dir = run_dir / "wandb" / "latest-run" / "files" / "media" / "images"

    if not images_dir.exists():
        return None, None

    preferred_prefixes = [
        "val_reconstructions",
        "train_reconstructions",
        "reconstructions",
    ]

    # 1) Exact target step, preferred prefix order
    for pref in preferred_prefixes:
        exact = sorted(images_dir.glob(f"{pref}_{target_step}_*.png"))
        if exact:
            return exact[-1], pref

    # 2) Any step, preferred prefix order (choose closest to target)
    for pref in preferred_prefixes:
        candidates = sorted(images_dir.glob(f"{pref}_*_*.png"))
        parsed = []
        for p in candidates:
            st = _extract_step_from_image_name(p)
            if st is not None:
                parsed.append((abs(st - target_step), st, p))
        if parsed:
            parsed.sort(key=lambda x: (x[0], -x[1]))
            return parsed[0][2], pref

    # 3) Any image in the folder
    any_png = sorted(images_dir.glob("*.png"))
    if any_png:
        return any_png[-1], "other"

    return None, None


def _row_title(row, prefix=""):
    return (
        f"{prefix}{row['run_label']}\n"
        f"psnr={row['psnr']:.3f}, ssim={row['ssim']:.4f}, mse={row['mse']:.6f}\n"
        f"disc={row['discriminator_choice']}, gdw={row['gen_disc_weight']}, "
        f"perc={row['perceptual_loss_weight']}, kl={row['kl_loss_weight']}"
    )


def _show_two_runs(row_a, row_b, target_step: int, title_a="Best", title_b="Worst"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, row, ttl in zip(axes, [row_a, row_b], [title_a, title_b]):
        img_path, img_kind = _find_media_image_for_row(row, target_step)
        ax.axis("off")
        if img_path is None:
            ax.set_title(f"{ttl} (no image found)\n{row['run_label']}")
            continue
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(_row_title(row, prefix=f"{ttl} | {img_kind}: "), fontsize=9)

    plt.tight_layout()
    plt.show()


def _show_best_per_factor_value(df_local, factor_col: str, target_step: int):
    if factor_col not in df_local.columns:
        print(f"Skip {factor_col}: missing column")
        return

    subset = df_local[df_local[factor_col].notna()].copy()
    if subset.empty:
        print(f"Skip {factor_col}: no non-null values")
        return

    # Best run inside each factor value by psnr/ssim/mse
    ranked = subset.sort_values([factor_col, "psnr", "ssim", "mse"], ascending=[True, False, False, True])
    best_per_value = ranked.groupby(factor_col, as_index=False).head(1).reset_index(drop=True)

    n = len(best_per_value)
    cols = 2
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
    axes = axes.flatten() if n > 1 else [axes]

    for i, (_, row) in enumerate(best_per_value.iterrows()):
        ax = axes[i]
        img_path, img_kind = _find_media_image_for_row(row, target_step)
        ax.axis("off")
        if img_path is None:
            ax.set_title(f"{factor_col}={row[factor_col]}\n(no image found)")
            continue
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(
            f"{factor_col}={row[factor_col]} | {img_kind}\n"
            + _row_title(row),
            fontsize=9,
        )

    # Hide unused subplots
    for j in range(n, len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"Best run per {factor_col} value @ step {target_step}", fontsize=13)
    plt.tight_layout()
    plt.show()


if "df" not in globals() or df.empty:
    print("`df` is missing or empty. Run the earlier aggregation cells first.")
else:
    ranked_all = df.sort_values(["psnr", "ssim", "mse"], ascending=[False, False, True]).reset_index(drop=True)
    best_row = ranked_all.iloc[0]
    worst_row = ranked_all.iloc[-1]

    print("=== Best vs Worst (visual) ===")
    _show_two_runs(best_row, worst_row, TARGET_STEP, title_a="Best overall", title_b="Worst overall")

    print("=== Ablation visuals: best per factor value ===")
    for factor in ["discriminator_choice", "perceptual_loss_weight", "kl_loss_weight", "gen_disc_weight"]:
        print(f"\n--- {factor} ---")
        _show_best_per_factor_value(df, factor, TARGET_STEP)
